# 《PythAPCS123》單元 13-7：輸出格式防禦與對齊心法（Presentation / Format WA 防範）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-7_presentation_error_and_output_formatting.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：徹底化解「演算法明明完全正確、答案也算得分毫不差，上傳 OJ 卻依然拿到 WA（Wrong Answer）」的千古奇冤！深刻理解現代競技程式線上評判系統底層「字元級比對（Diff）」的嚴苛規則，地毯式剷除五大常見排版格式硬傷：`input()` 殘留提示文字污染 stdout、每行末尾多印出多餘空白（Trailing Space）、整數運算誤印出浮點小數點（如 `4.0` vs `4`）、大小寫與布林字串規格不符（如 `YES` vs `Yes`、`true` vs `True`）、以及多行輸出錯位與末行換行偏差。學會運用解包 `print(*arr)` 與 f-string 打造毫釐不差的乾淨輸出防線。


### 13.7.1 線上評判對輸出的字元級苛求（嚴禁多一個空格、少一個換行、多餘標點）

在學校的作業或人工閱卷中，如果題目的答案是 `42`，而你印出了 `答案是 42`，老師通常會認可你算對了；或者即便你在數字後面多按了一個空格，人類肉眼甚至根本察覺不出來。

但在競技程式（如 APCS、ZeroJudge、Codeforces、LeetCode）的自動評判系統中，**裁判是冷酷無情的機器，而非寬容的人類**！
評判系統檢查你的答案時，底層使用的是類似作業系統級別的 `diff`（檔案文字逐位比對）工具：
- 它會將你的程式透過標準輸出（`sys.stdout`）列印出來的「每一個字元、每一個空白鍵（ASCII 32）、每一個換行符號（ASCII 10）」，與官方預先準備的標準解答（Output Standard）進行**「字元級二進位嚴格比對」**！
- 只要官方標程結尾沒有空白，而你多印了一個空格；或者官方要求輸出全大寫的 `"YES"`，而你印出了首字大寫的 `"Yes"`；甚至官方要求印出整數 `15`，而你印出了浮點數 `15.0`——評判系統都會立刻判定「比對失敗」，無情記上一記刺眼的 **WA（Wrong Answer）**！

過去有些舊型 OJ 會設立「PE（Presentation Error，格式錯誤）」狀態碼提醒考生，但**現代多數評判系統（包括 APCS 檢定系統）已將所有格式瑕疵直接歸類為 WA**！因此，掌握字元級的精確輸出排版，是確保所有辛苦寫出的演算法能順利換算為真金白銀分數的關鍵臨門一腳。


In [ ]:
# 13.7.1 程式碼演示：模擬 OJ 字元級比對差異（看似相同卻被判 WA）
import io
import sys

# 模擬官方標準解答答案串流
official_answer = "10 20 30\n"

# 模擬學生輸出的三種常見微小瑕疵
student_a = "10 20 30 \n" # 瑕疵 A: 行末多了一個不可見的空格
student_b = "10 20 30"    # 瑕疵 B: 結尾少了一個換行符號
student_c = "10  20 30\n"  # 瑕疵 C: 數字之間多敲了一個空格

def judge_diff(student_out, official_out, label):
    print(f"\n--- 評判學生 [{label}] 輸出結果 ---")
    print(f"學生輸出內容: {repr(student_out)}")
    print(f"官方標程內容: {repr(official_out)}")
    if student_out == official_out:
        print(" verdict: ✅ AC (Accepted) - 字元完全吻合！")
    else:
        print(" verdict: ❌ WA (Wrong Answer) - 存在字元級差異！")
        # 顯示差異位置
        for i in range(max(len(student_out), len(official_out))):
            ch_s = student_out[i] if i < len(student_out) else "<EOF>"
            ch_o = official_out[i] if i < len(official_out) else "<EOF>"
            if ch_s != ch_o:
                print(f"  --> 第一次分歧發生於第 {i} 個字元: 學生為 {repr(ch_s)}，官方要求為 {repr(ch_o)}")
                break

judge_diff(student_a, official_answer, "學生 A (末尾多空格)")
judge_diff(student_b, official_answer, "學生 B (少換行)")
judge_diff(student_c, official_answer, "學生 C (中間多空格)")


### 13.7.1 語法重點回顧與核心觀念提煉

字元級比對的防禦心智模型：
1. **空白與換行皆是實體字元**：在電腦內部，空格 `' '`（ASCII 32）和文字 `'A'` 一樣佔據一個位元組；換行符號 `'\n'`（ASCII 10）也是實體字元。永遠不要把空白當作「什麼都沒有」。
2. **無情對齊原則**：題目規定輸出什麼，就「不多一個字元、不少一個字元、不增添任何修飾語」。
3. **善用 `repr()` 檢查肉眼盲區**：在本機除錯時，如果懷疑格式出錯，可以使用 `print(repr(ans))` 將隱藏的空白、換行與單引號完整顯形！


In [ ]:
# 13.7.1 學生實作練習：字串嚴格一致性比對器
# 任務說明：實作 verify_exact_match(output_str, expected_str) 函式
# 模擬裁判系統的比對機制：
# 若 output_str 與 expected_str 的長度與每一個字元完全嚴格吻合，回傳 True
# 若有任何一個字元、空格或換行不一致，回傳 False

def verify_exact_match(output_str: str, expected_str: str) -> bool:
    # 請在此處進行嚴格完全比對
    return output_str == expected_str

# 測試用例
print("完全吻合:", verify_exact_match("Result: 100\n", "Result: 100\n"))
print("末尾多空格:", verify_exact_match("Result: 100 \n", "Result: 100\n"))


In [ ]:
# 13.7.1 單元測試驗證
assert verify_exact_match("AC\n", "AC\n") == True
assert verify_exact_match("AC", "AC\n") == False
assert verify_exact_match("1 2 3", "1 2 3 ") == False
assert verify_exact_match("", "") == True
print("13.7.1 單元測試全數通過！")


### 13.7.2 考場最冤枉 WA：input("請輸入數字：") 殘留提示文字造成全盤皆輸

在剛開始學習 Python 時，許多教科書或初學者教程為了友善互動，常教大家這樣寫：
```python
# 致命考場毒藥：千萬不要在 input() 括號內放入提示文字！
n = int(input("請輸入數字 n: "))
```

當你在自己的電腦上執行時，這看起來非常親切溫馨。然而，在 APCS 或任何 Online Judge 系統上，這行代碼會直接造成**「整份程式全盤皆輸、所有測試點全部 WA」**的慘烈悲劇！

這是因為 Python 的 `input(prompt)` 機制是：**它會把括號內的字串 `prompt` 直接寫入「標準輸出（stdout）」**！
換言之，裁判系統在收集你程式的標準輸出時，原本預期只會收到計算結果 `42`，結果卻收到了：
```
請輸入數字 n: 42
```
裁判系統拿著 `"請輸入數字 n: 42"` 去跟官方標程的 `"42"` 比對，第一秒就判定字元完全不符，整場考試直接掛蛋！

**考場第一鐵律：在競技程式與 APCS 考場中，所有的 `input()` 必須「永遠保持絕對乾淨的空括號」：`input()`！嚴禁放入任何一個漢字或提示英文！**


In [ ]:
# 13.7.2 程式碼演示：input 提示字元對標準輸出的污染現場
import io
import sys

# 模擬標準輸入串流
fake_stdin = io.StringIO("100\n")
# 建立捕捉標準輸出的串流
fake_stdout = io.StringIO()

old_stdin = sys.stdin
old_stdout = sys.stdout

sys.stdin = fake_stdin
sys.stdout = fake_stdout

# 錯誤示範：初學者寫法
try:
    val = int(input("請輸入一個整數："))
    print(val * 2)
finally:
    sys.stdin = old_stdin
    sys.stdout = old_stdout

# 檢查標準輸出中被塞進了什麼內容
captured_output = fake_stdout.getvalue()
print("=== 裁判系統實際截獲的標準輸出內容 ===")
print(repr(captured_output))
print("\n解剖分析：")
print("原意只想輸出結果 '200\\n'")
print("但因為 input 帶提示文字，導致實際輸出變成 '請輸入一個整數：200\\n'")
print("結果：裁判系統 diff 比對宣告失敗，判定 WA 0 分！")


### 13.7.2 語法重點回顧與核心觀念提煉

輸入提示詞防禦金律：
1. **空括號是唯一的合法形態**：
   - 錯誤寫法：`n = int(input("請輸入測資："))` ❌ (必拿 WA)
   - 正確寫法：`n = int(input())` ✅ (乾淨無暇)
2. **多個整數讀取標準形態**：
   ```python
   a, b = map(int, input().split()) # 永遠不要加引號提示
   ```
3. **本機練習即刻矯正**：平常在練習任何解題時，就要徹底戒除寫 prompt 的習慣，讓代碼完全符合競賽規範。


In [ ]:
# 13.7.2 學生實作練習：乾淨輸入讀取與平方輸出
# 任務說明：實作 clean_input_and_square() 函式
# 從標準輸入讀取一個整數（務必使用不帶任何提示文字的乾淨 input()！）
# 計算該整數的平方並回傳計算結果！

def clean_input_and_square() -> int:
    # 請使用純淨的 input() 讀取並轉型
    val = int(input())
    return val ** 2

# 測試用例（模擬輸入 7）
import io, sys
old_stdin = sys.stdin
sys.stdin = io.StringIO("7\n")
res = clean_input_and_square()
sys.stdin = old_stdin
print("平方計算結果:", res)


In [ ]:
# 13.7.2 單元測試驗證
import io, sys
old_in = sys.stdin

sys.stdin = io.StringIO("12\n")
assert clean_input_and_square() == 144

sys.stdin = io.StringIO("-5\n")
assert clean_input_and_square() == 25

sys.stdin = old_in
print("13.7.2 單元測試全數通過！")


### 13.7.3 行末多餘空白（Trailing Space）防範：print(*ans) 與 ' '.join(ans)

在 APCS 許多題目中，題目要求輸出一個數列，各元素之間以空白隔開，例如：`10 20 30 40`。
許多初學者在寫迴圈輸出時，最直覺的做法是：
```python
# 潛在格式地雷：行末多餘空白（Trailing Space）
for x in ans:
    print(x, end=" ") # 每個數字後面都印一個空格
print()
```

請仔細觀察這段代碼最後產生的字串：它實際印出的是 `"10 20 30 40 "`！注意看最後一個數字 `40` 的後面，**硬生生多掛了一個多餘的空白鍵（ASCII 32）**！
在要求極度嚴苛的評判測資中，這個行末多餘空格就會被判定與標準解答不吻合，痛失該測試點的分數！

針對此問題，Python 提供了兩大極具美感且絕無行末空格的**「考場雙神技」**：
1. **神技一：解包語法 `print(*ans)`**
   若 `ans` 是一個串列（如 `ans = [10, 20, 30]`），直接寫 `print(*ans)`！
   星號 `*` 會將串列中的元素逐一解包傳入 `print()`，而 `print` 預設的間隔符號就是單一空白（`sep=" "`），更重要的是：**它只會在元素「之間」插入空白，最後一個元素的後方絕對不會有多餘空格，並直接補上換行！**
2. **神技二：字串拼接 `' '.join(...)`**
   若元素已是字串，使用 `' '.join(ans)` 拼接；若為數字，使用 `' '.join(map(str, ans))`，乾淨俐落！


In [ ]:
# 13.7.3 程式碼演示：行末多餘空格 vs print(*ans) 乾淨解包
ans_list = [10, 20, 30, 40]

print("--- 壞習慣示範：迴圈 end=' ' 造成行末殘留空格 ---")
import io, sys
buf_bad = io.StringIO()
old_out = sys.stdout
sys.stdout = buf_bad

for x in ans_list:
    print(x, end=" ")
print()

sys.stdout = old_out
bad_str = buf_bad.getvalue()
print("實際輸出字串:", repr(bad_str))
print("觀察：注意 '40 ' 後方多了一個多餘空格！❌")

print("\n--- 好習慣示範：使用 print(*ans) 解包 ---")
buf_good = io.StringIO()
sys.stdout = buf_good

# 考場一秒搞定神技：
print(*ans_list)

sys.stdout = old_out
good_str = buf_good.getvalue()
print("實際輸出字串:", repr(good_str))
print("觀察：'40\\n' 後方直接換行，完全沒有多餘空格！✅")


### 13.7.3 語法重點回顧與核心觀念提煉

輸出單行空格分隔序列的兩大王牌：
1. **數字串列首選**：`print(*ans)`
   - 語法最短（只需 12 個字元）。
   - 自動在元素間補單一空格，末尾自動換行，無行末空格瑕疵。
2. **自訂間隔符號**：
   - 若題目要求用逗號分隔（如 `1,2,3`）：`print(*ans, sep=",")`
   - 若題目要求不換行且無空格：`print(*ans, sep="")`
3. **除錯自檢**：在上傳前，在本機終端機用滑鼠圈選最後一行的結尾，看看游標能否反白出多餘的空格；或者使用 `repr()` 檢查字串結尾。


In [ ]:
# 13.7.3 學生實作練習：無瑕疵空格數列格式化
# 任務說明：實作 format_space_separated(arr) 函式
# 傳入整數串列 arr，將其格式化為「各數字以單一空白隔開」的字串
# 嚴格要求：最後一個數字後方絕對不可有多餘的空格！
# 若傳入空串列 []，回傳空字串 ""

def format_space_separated(arr: list) -> str:
    # 請在此處使用 ' '.join 或類似手法完成無瑕疵格式化
    if not arr:
        return ""
    return " ".join(map(str, arr))

# 測試用例
print("格式化結果:", repr(format_space_separated([5, 15, 25, 35])))
print("空串列測試:", repr(format_space_separated([])))


In [ ]:
# 13.7.3 單元測試驗證
assert format_space_separated([1, 2, 3]) == "1 2 3"
assert format_space_separated([42]) == "42"
assert format_space_separated([]) == ""
assert format_space_separated([10, 20]) == "10 20"
# 驗證末尾不包含多餘空格
res_text = format_space_separated([10, 20])
assert not res_text.endswith(" ")
print("13.7.3 單元測試全數通過！")


### 13.7.4 數值型態輸出對齊：整數要求印出 4 誤印為 4.0、浮點數格式化輸出小數後幾位

在數值計算題中，題目常有嚴格的數值輸出型態規範，例如：
- 「請輸出計算後的平均整數商數」
- 「請輸出結果，四捨五入輸出至小數點後第 2 位」

這裡潛伏著兩種極易導致 WA 的型態對齊陷阱：

#### 陷阱一：整數題目誤印出小數點（`4` vs `4.0`）
在 Python 3 中，一般的單斜線除法 `/` 無論能否整除，其回傳型態永遠都是浮點數 `float`。
例如 `8 / 2` 的計算結果不是整數 `4`，而是浮點數 `4.0`！
如果題目的範例輸出寫著 `4`，而你印出了 `4.0`，在字元比對上多了小數點和零，會直接被評判系統判為 WA！
- *防禦對策*：題目若要求整數，除法運算請堅決使用「雙斜線整數除法 `//`」；或者在最後輸出前強制轉型為 `int(round(val))`。

#### 陷阱二：浮點數位數未精確對齊
若題目要求輸出到小數點後 2 位（如 `3.50`），如果你直接寫 `round(3.5, 2)` 並印出，Python 會輸出 `3.5`（尾隨的 0 會被自動省略），直接導致格式不合而被判 WA！
- *防禦對策*：使用 **f-string 格式化字串 `f"{val:.2f}"`**！它會自動補齊末尾的零（例如將 `3.5` 補成 `"3.50"`），並精準控制小數點後的位數。


In [ ]:
# 13.7.4 程式碼演示：整數除法 vs 浮點除法輸出，以及 f-string 位數補齊
print("--- 陷阱 1: 4 vs 4.0 型態字元差異 ---")
val_float = 8 / 2   # 單斜線產生 float: 4.0
val_int = 8 // 2    # 雙斜線產生 int: 4

print(f"單斜線 8 / 2 輸出: '{val_float}' (長度 {len(str(val_float))})")
print(f"雙斜線 8 // 2 輸出: '{val_int}' (長度 {len(str(val_int))})")
print(f"兩者字元比對是否相等: {str(val_float) == str(val_int)} ❌ (WA 殺手！)")

print("\n--- 陷阱 2: round() 尾隨零遺失 vs f-string 嚴格補齊 ---")
target_val = 3.5
# 題目要求：輸出至小數點後兩位，官方期望 "3.50"
round_out = str(round(target_val, 2)) # 產出 "3.5"，末尾零丟失！
fstring_out = f"{target_val:.2f}"     # 產出 "3.50"，完美精確對齊！

print(f"使用 round(x, 2) 輸出: '{round_out}' ❌ (缺末尾 0)")
print(f"使用 f'{{x:.2f}}' 輸出: '{fstring_out}' ✅ (完美對齊官方格式)")


### 13.7.4 語法重點回顧與核心觀念提煉

數值格式對齊的黃金法則：
1. **整數題目必用 `//`**：求平均、求商數、求整數數量，運算過程請維持純整數運算；若中間牽涉開根號或浮點運算，輸出前以 `int(round(ans))` 還原為整數。
2. **小數位數一律使用 `f"{x:.Nf}"`**：
   - 小數點後一位：`f"{x:.1f}"`
   - 小數點後兩位：`f"{x:.2f}"`
   - 小數點後四位：`f"{x:.4f}"`
3. **千分位逗號注意**：除非題目明確要求印出 `1,000`，否則不可在 f-string 中加入逗號格式化。


In [ ]:
# 13.7.4 學生實作練習：成績平均與格式對齊器
# 任務說明：實作 format_grade_report(scores) 函式
# 傳入非空整數串列 scores
# 需求：
# 1. 計算整數總分 total（以整數型態輸出，如 "Total: 250"）
# 2. 計算浮點平均 avg，格式化精準至小數點後兩位（如 "Average: 83.33"，若為 80 需補為 "80.00"）
# 3. 回傳格式化字串：f"Total: {total}, Average: {avg_str}"

def format_grade_report(scores: list) -> str:
    # 請在此處實作精準格式對齊
    total = sum(scores)
    avg = total / len(scores)
    return f"Total: {total}, Average: {avg:.2f}"

# 測試用例
print("測試報告 1:", format_grade_report([80, 80, 80]))
print("測試報告 2:", format_grade_report([100, 95, 88]))


In [ ]:
# 13.7.4 單元測試驗證
assert format_grade_report([80, 80, 80]) == "Total: 240, Average: 80.00"
assert format_grade_report([100, 50]) == "Total: 150, Average: 75.00"
assert format_grade_report([10, 20, 25]) == "Total: 55, Average: 18.33"
print("13.7.4 單元測試全數通過！")


### 13.7.5 字串大小寫與真假值對齊：YES/NO vs Yes/No, True/False vs true/false

在判定類題目中，題目通常會要求：若符合條件輸出判定字串，否則輸出相反字串。
初學者最常犯的粗心失誤，就是**「大小寫規格沒有看清楚」**！

在電腦內部，大寫字母與小寫字母的 ASCII 編碼完全不同：
- `'Y'` 的 ASCII 碼是 89
- `'y'` 的 ASCII 碼是 121
兩者在二進位層面上完全不同！常見的翻車組合包括：
1. **YES / NO 系列**：
   - 題目要求全大寫：`YES` / `NO`
   - 考生隨手印成首字大寫：`Yes` / `No`
   - 考生隨手印成全小寫：`yes` / `no`
2. **真假值布林系列**：
   - 在 Python 中，直接 `print(True)` 會輸出首字大寫的 `"True"`。
   - 但若題目是從 C++ 或 Java 翻譯過來的題目，題意規範輸出全小寫的 `"true"` 或 `"false"`，若直接寫 `print(is_valid)` 就會當場 WA！
3. **無解提示詞**：例如題目要求輸出全大寫的 `IMPOSSIBLE` 或 `-1`，打成 `Impossible` 或 `None`。

面對這類判定題，最佳防禦習慣是：**直接從題目的「輸出格式」或「範例輸出」中，用滑鼠複製該單字**，貼到代碼的引號中，徹底杜絕大小寫手滑！


In [ ]:
# 13.7.5 程式碼演示：布林真值轉換與大小寫嚴格對齊
def boolean_to_judgement(is_valid: bool, format_type: str) -> str:
    # 支援競賽常見的三種規格
    if format_type == "UPPER":
        return "YES" if is_valid else "NO"
    elif format_type == "TITLE":
        return "Yes" if is_valid else "No"
    elif format_type == "LOWER_BOOL":
        return "true" if is_valid else "false" # C/C++ 風格全小寫
    elif format_type == "PYTHON_BOOL":
        return "True" if is_valid else "False" # 原生 Python 風格
    return "UNKNOWN"

test_val = True
print("--- 相同真值在不同題意要求下的輸出字串對比 ---")
print("題目要求全大寫:", repr(boolean_to_judgement(test_val, "UPPER")))
print("題目要求首字大寫:", repr(boolean_to_judgement(test_val, "TITLE")))
print("題目要求全小寫布林:", repr(boolean_to_judgement(test_val, "LOWER_BOOL")))
print("Python 原生輸出:", repr(boolean_to_judgement(test_val, "PYTHON_BOOL")))
print("警告：這四種字串在裁判系統眼中互不相等，大小寫弄錯一律判定 WA！")


### 13.7.5 語法重點回顧與核心觀念提煉

字串規格防禦三步檢核法：
1. **複製代替手打**：在題目的「輸出說明」中找到目標關鍵字（例如 `VALID`、`NO`、`None`），用滑鼠直接複製，避免拼字錯誤與大小寫偏差。
2. **全小寫布林轉換技巧**：若題目要求輸出小寫的 `true` / `false`：
   ```python
   print(str(flag).lower()) # 將 True 轉為 "true"
   ```
3. **檢查空輸出規格**：題目若規定「無符合項目時輸出 -1」或「輸出 None」，務必設立獨立分支，不可讓變數為空或回傳空行。


In [ ]:
# 13.7.5 學生實作練習：質數判定格式對齊器
# 任務說明：實作 judge_prime_output(n, style) 函式
# 判斷正整數 n 是否為質數（大於 1 且除 1 與本身外無其他因數）
# 依據 style 參數嚴格對齊輸出規格：
# - style == "YES_NO": 質數回傳 "YES"，非質數回傳 "NO"
# - style == "LOWER": 質數回傳 "true"，非質數回傳 "false"
# - style == "TITLE": 質數回傳 "Prime"，非質數回傳 "Not Prime"

def judge_prime_output(n: int, style: str) -> str:
    # 1. 先判斷是否為質數
    is_prime = True
    if n < 2:
        is_prime = False
    else:
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                is_prime = False
                break
                
    # 2. 依據規格精準輸出
    if style == "YES_NO":
        return "YES" if is_prime else "NO"
    elif style == "LOWER":
        return "true" if is_prime else "false"
    elif style == "TITLE":
        return "Prime" if is_prime else "Not Prime"
    return ""

# 測試用例
print("7 的 YES_NO 規格:", judge_prime_output(7, "YES_NO"))
print("4 的 LOWER 規格:", judge_prime_output(4, "LOWER"))


In [ ]:
# 13.7.5 單元測試驗證
assert judge_prime_output(7, "YES_NO") == "YES"
assert judge_prime_output(9, "YES_NO") == "NO"
assert judge_prime_output(2, "LOWER") == "true"
assert judge_prime_output(1, "LOWER") == "false"
assert judge_prime_output(13, "TITLE") == "Prime"
assert judge_prime_output(15, "TITLE") == "Not Prime"
print("13.7.5 單元測試全數通過！")


### 13.7.6 多行輸出與最後一筆換行規範：防止行數錯位與殘留空行

在處理多筆測資（Multi-testcases）或矩陣網格列印時，輸出的「行數」與「換行符號（`\n`）」的對齊是最後一處關鍵防線。

常見的兩大換行災難：
1. **行數錯位與漏換行**：
   在處理多筆輸入時，每一筆測資算完後，必須確保呼叫一次帶換行的 `print()`。如果忘記換行，會導致第二筆測資的輸出直接黏在第一筆的屁股後面（例如第一筆答案是 `10`，第二筆是 `20`，黏在一起變成 `1020`，整盤 WA）。
2. **行與行之間的空行（Blank Line）要求**：
   有些題目會特別註明：**「每組測試資料的輸出之間，請輸出一行空白行；但『最後一組資料後方不得有空白行』。」**
   初學者如果每次都在迴圈最後加印 `print()`，最後一組資料就會殘留一個不被允許的空行，再次被判定 WA！
   - *正解*：使用旗標變數（如 `first_case = True`）或將所有答案組裝於串列中，最後使用 `"\n\n".join(all_outputs)` 一次性印出！

最後一筆資料的末尾是否該換行？
在多數現代 OJ 與 APCS 評判中，**每一行的結尾（包括最後一行）都應該有正常的單一換行符號（`\n`）**（這是 POSIX 標準文字檔的定義）。Python 的 `print()` 預設結尾就自帶一個 `\n`，因此只要正常使用 `print()`，末尾自然符合規範。


In [ ]:
# 13.7.6 程式碼演示：案例間空行分隔之標準乾淨模板
# 需求：輸出多個區塊，區塊與區塊之間有一行空行，但最後一個區塊後方不可有空行

blocks = [
    ["Case #1:", "Result A", "Result B"],
    ["Case #2:", "Result C", "Result D"],
    ["Case #3:", "Result E", "Result F"]
]

print("--- 錯誤做法：盲目在每組後方 print()，造成末尾殘留空行 ---")
import io, sys
buf_bad = io.StringIO()
old_out = sys.stdout
sys.stdout = buf_bad

for b in blocks:
    for line in b:
        print(line)
    print() # 盲目印空行，最後一組後方也會多出一個空行！

sys.stdout = old_out
print("錯誤做法結尾字元:", repr(buf_bad.getvalue()[-4:]))

print("\n--- 正確做法：使用旗標變數控制空行分隔 ---")
buf_good = io.StringIO()
sys.stdout = buf_good

first = True
for b in blocks:
    if not first:
        print() # 只有從第二組開始，才在前方補空行！
    first = False
    for line in b:
        print(line)

sys.stdout = old_out
good_res = buf_good.getvalue()
print("正確做法結尾字元:", repr(good_res[-4:]))
print("輸出預覽:\n" + good_res)


### 13.7.6 語法重點回顧與核心觀念提煉

多行與空行控制的黃金原則：
1. **旗標變數前置隔開法（First-flag Pattern）**：
   ```python
   first = True
   for case in cases:
       if not first:
           print() # 兩組之間的空行
       first = False
       print(case_result)
   ```
2. **二維網格走訪印出模板**：
   ```python
   for row in grid:
       print(*row) # 每列內部空格隔開，每列結束自動換行
   ```
3. **保持冷靜**：只要把輸出排版當成演算法的一部分嚴格對待，就能確保 100% 的計算心血都能轉換為實實在在的通過狀態（AC）！


In [ ]:
# 13.7.6 學生實作練習：二維網格乾淨列印器
# 任務說明：實作 format_grid_output(grid) 函式
# 傳入二維整數串列 grid（如 [[1, 2], [3, 4]]）
# 回傳一個多行字串：
# 1. 每一列的數字之間以單一空白隔開
# 2. 每一列結束時換行
# 3. 各列末尾不可有多餘空格

def format_grid_output(grid: list) -> str:
    # 請在此處實作乾淨二維網格格式化
    lines = []
    for row in grid:
        lines.append(" ".join(map(str, row)))
    return "\n".join(lines)

# 測試用例
test_grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("格式化網格輸出:\n" + format_grid_output(test_grid))


In [ ]:
# 13.7.6 單元測試驗證
g = [[1, 2], [3, 4]]
expected = "1 2\n3 4"
assert format_grid_output(g) == expected

g_single = [[42]]
assert format_grid_output(g_single) == "42"

g_empty = []
assert format_grid_output(g_empty) == ""
print("13.7.6 單元測試全數通過！")


## 13.7 總結與輸出格式防禦查核表

在本單元中，我們揭開了 Online Judge 自動評判系統「字元級 diff 比對」的底層機制，並學會了消滅所有格式瑕疵的精確對齊技術。送出代碼前，請逐一檢查以下輸出格式查核清單：

| 格式地雷項目 | 常見翻車現場 | 考場標準 AC 防禦模板 |
| :--- | :--- | :--- |
| **`input()` 提示詞污染** | `input("請輸入：")` 導致提示詞被印到標準輸出 | 嚴格保持純淨空括號：`input()` |
| **行末多餘空格** | 迴圈結尾 `end=" "` 導致行末多一個空白 | 使用解包神技：`print(*ans)` 或 `' '.join(...)` |
| **整數印成浮點數** | 誤用單斜線除法印出 `4.0` 而非 `4` | 整數題目一律使用整數除法 `//` 或 `int()` 轉型 |
| **浮點位數未對齊** | `round(3.5, 2)` 印出 `3.5` 丟失末尾零 | 強制使用 f-string 嚴格控制小數位數：`f"{val:.2f}"` |
| **字串大小寫手滑** | 題目要 `YES` 誤打為 `Yes`、要 `true` 打為 `True` | **直接從題目說明中複製目標單字**，絕不手打猜測 |
| **多筆測資隔行空行** | 盲目在末尾加印導致最後一筆殘留空行 | 採用「旗標前置隔開法（`if not first: print()`）」 |

### 🚀 下一步學習指引
至此，我們已經把所有可能在編譯期、執行期、邏輯期、效能期與格式期引爆的錯誤全數拆解完畢！
然而，在真實的 APCS 考場上，面對長達 2.5 小時的緊迫考試、陌生的電腦環境與緊張的心情，一旦程式碼出錯，最忌諱的就是「慌亂地亂改一通，越改越糟」。
在第十三章的最終決戰單元 **13-8《考場系統化除錯戰略：錯誤重現、二分隔離、斷言與送出前 SOP》** 中，我們將把所有除錯技巧整合為一套標準化的考場應變作業流程（SOP），助你冷靜從容地奪下滿分！
